# Sultai local mana-base search

Start from the explicit example mana base and improve it with legal one-land replacements.
Each neighborhood uses the existing exact scalar evaluator, with no global grid or random restarts.
The example spell profiles and starting base come from `examples.sultai`.

Run all cells. Edit the domain or rollout settings and recreate the evaluator when those change.
For different penalties or budgets, rerun the search cell: cached draw calculations remain available,
and each search starts from `INITIAL_BASE`. Budget counts distinct bases visited in that run,
including cached bases and the starting base. A budget stop does not certify a local optimum.

Scores retain the current optimistic independent-casting-opportunity model: spells do not compete
for mana, land drops are not sequenced, and Cavern restrictions are excluded. Mode weights split
physical spell copies as in the existing evaluator. Penalties are illustrative per-land costs.

In [1]:
from pathlib import Path
import sys
from collections import defaultdict
from html import escape
from IPython.display import HTML, display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mana_search.py').exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from examples.sultai import MANA_BASE, SPELLS
from mana_lands import ManaSource
from mana_search import ManaSpace, LocalManaEvaluator, local_search

def name(source):
    return f'{source.kind}:{source.colors}'

def lands(entries):
    return ', '.join(f'{n} {name(s)}' for s, n in entries) or 'none'

def show(headers, rows):
    head = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

## Domain and structured evaluation

The domain uses the catalog's dated Standard snapshot; it does not fetch current legality.
Four Caverns and one Swamp occupy five of the 26 slots. Source bounds define eligibility;
the starting base is supplied explicitly and is validated before any search. Optional source
families can be added here. Opportunities remain split by casting cost, turn, and play/draw;
mana value is a reporting grouping, so different colored costs remain distinct.

In [2]:
LAND_COUNT = 26
COLORS = 'UBG'
FORMAT = 'standard'
LAND_FAMILIES = {
    'basic': (0, 1), 'shock': (0, 4), 'fast': (0, 4),
    'verge': (0, 4), 'town': (0, 4),
}
FIXED_LANDS = {ManaSource('WUBRG', 'cavern'): 4, ManaSource('B', 'basic'): 1}
INITIAL_BASE = dict(MANA_BASE)
TURNS = 7
POSITION = 'both'
OPENING_HAND_SIZE = 7

space = ManaSpace.from_colors(LAND_COUNT, COLORS, families=LAND_FAMILIES,
                             format=FORMAT, fixed=FIXED_LANDS)
space.counts(INITIAL_BASE)
evaluator = LocalManaEvaluator(space, SPELLS, turns=TURNS, position=POSITION,
                               opening_hand_size=OPENING_HAND_SIZE)
show(['Source', 'Minimum', 'Maximum', 'Initial'],
     [(name(s), lo, hi, INITIAL_BASE.get(s, 0)) for s, (lo, hi) in space.bounds.items()])
print(f'{LAND_COUNT} lands; {len(space.sources)} source types. No global candidates or grid constructed.')

Source,Minimum,Maximum,Initial
basic:B,1,1,1
basic:G,0,1,0
basic:U,0,1,0
cavern:WUBRG,4,4,4
fast:BG,0,4,0
fast:UG,0,4,0
shock:BG,0,4,4
shock:UB,0,4,4
shock:UG,0,4,4
town:WUBRG,0,4,4


26 lands; 13 source types. No global candidates or grid constructed.


## Search from the supplied base

Each iteration scores the legal one-land replacements and accepts the best strict improvement.
Equal improving scores use canonical source-count order; equal-score moves are not accepted.
If the budget ends during a neighborhood, the best evaluated replacement is returned.
The progress lines report completed/partial neighborhoods; the final status distinguishes a
budget stop from a fully checked one-land local optimum. Multi-land improvements may still exist.

In [3]:
SHOCK_PENALTY = 0.012
TOWN_PENALTY = 0.015
EVALUATION_BUDGET = 500

cached_before = len(evaluator.cache)
result = local_search(
    evaluator, INITIAL_BASE, shock_penalty=SHOCK_PENALTY, town_penalty=TOWN_PENALTY,
    evaluation_budget=EVALUATION_BUDGET,
    progress=lambda n, best: print(f'{n:4d} unique bases visited; best score {float(best.score):.8f}'),
)
print(f'Stopped: {result.stopped}')
print(f'{result.evaluations} unique bases visited; {len(evaluator.cache) - cached_before} new draw evaluations; '
      f'{result.seconds:.2f} seconds')
show(['Base', 'Raw opportunities', 'Penalty', 'Score', 'Delta from initial'], [
    ('Initial', float(result.baseline.raw_score), float(result.baseline.penalty), float(result.baseline.score), 0),
    ('Result', float(result.best.raw_score), float(result.best.penalty), float(result.best.score), float(result.best.delta)),
])
show(['Move', 'OUT', 'IN', 'Score', 'Improvement'],
     [(i, lands(m.swap.out), lands(m.swap.incoming), f'{float(m.score):.8f}', f'{float(m.delta):+.8f}')
      for i, m in enumerate(result.moves, 1)])
show(['Source', 'Initial', 'Result', 'Change'],
     [(name(s), a, b, b-a) for s, a, b in zip(space.sources, result.baseline.counts, result.best.counts)])

  47 unique bases visited; best score 3.02643240
  87 unique bases visited; best score 3.03069241
 131 unique bases visited; best score 3.03402103
 176 unique bases visited; best score 3.03584873
 220 unique bases visited; best score 3.03633469
 277 unique bases visited; best score 3.03670819
 326 unique bases visited; best score 3.03775190
 375 unique bases visited; best score 3.03821459
 424 unique bases visited; best score 3.03849236
 468 unique bases visited; best score 3.03849236
Stopped: one-land local optimum
468 unique bases visited; 468 new draw evaluations; 216.23 seconds


Base,Raw opportunities,Penalty,Score,Delta from initial
Initial,3.2237459120991594,0.204,3.019745912099159,0
Result,3.158492364605696,0.12,3.038492364605696,0.018746452506536657


Move,OUT,IN,Score,Improvement
1,1 shock:UG,1 verge:BG,3.02643240,+0.00668649
2,1 shock:UB,1 basic:U,3.03069241,+0.00426001
3,1 shock:UG,1 verge:UG,3.03402103,+0.00332862
4,1 shock:UG,1 basic:G,3.03584873,+0.00182771
5,1 shock:BG,1 fast:BG,3.03633469,+0.00048595
6,1 shock:BG,1 shock:UB,3.03670819,+0.00037350
7,1 verge:UB,1 verge:BG,3.03775190,+0.00104372
8,1 shock:BG,1 fast:BG,3.03821459,+0.00046269
9,1 shock:BG,1 fast:BG,3.03849236,+0.00027777


Source,Initial,Result,Change
basic:B,1,1,0
basic:G,0,1,1
basic:U,0,1,1
cavern:WUBRG,4,4,0
fast:BG,0,3,3
fast:UG,0,0,0
shock:BG,4,0,-4
shock:UB,4,4,0
shock:UG,4,1,-3
town:WUBRG,4,4,0


## Opportunity changes

These tables describe the result versus the starting base. Expected copies retain the
spell-mode shares. Weighted contributions include spell importance; they exclude composition
penalties. Every cell contribution is divided by the total number of turn/position cells,
so summing a table's weighted deltas reproduces the raw-score change. Play and draw are
also shown separately. Cost groups retain their mana value as an additional label.

In [4]:
before = evaluator.components(INITIAL_BASE)
after = evaluator.components(dict(zip(space.sources, result.best.counts)))
divisor = TURNS * (2 if POSITION == 'both' else 1)

def mana_value(cost):
    return int(cost.rstrip('WUBRG') or 0) + sum(cost.count(c) for c in 'WUBRG')

def comparison(key, headers):
    groups = defaultdict(lambda: [0.0, 0.0, 0.0, 0.0])
    for index, components in enumerate((before, after)):
        for row in components.opportunities:
            values = groups[key(row)]
            values[index] += float(row.expected_copies) / divisor
            values[index + 2] += row.weighted_contribution / divisor
    show(headers + ['Initial copies', 'Result copies', 'Copy delta', 'Weighted delta'],
         [(*k, f'{v[0]:.8f}', f'{v[1]:.8f}', f'{v[1]-v[0]:+.8f}', f'{v[3]-v[2]:+.8f}')
          for k, v in sorted(groups.items())])

comparison(lambda r: (mana_value(r.cost), r.cost), ['MV', 'Cost'])
comparison(lambda r: (mana_value(r.cost),), ['MV'])
comparison(lambda r: (r.turn, r.position), ['Turn', 'Position'])
assert result.best.score >= result.baseline.score
assert result.evaluations <= EVALUATION_BUDGET
assert abs(sum(r.weighted_contribution for r in after.opportunities) / divisor
           - float(result.best.raw_score)) < 1e-12

MV,Cost,Initial copies,Result copies,Copy delta,Weighted delta
1,B,0.68422336,0.67761241,-0.00661095,-0.00661095
2,1B,0.45587831,0.45154747,-0.00433083,-0.00433083
2,BB,0.22916235,0.22077299,-0.00838936,-0.00838936
2,GG,0.10915946,0.10767122,-0.00148823,-0.00148823
2,UU,0.22944012,0.21959124,-0.00984888,-0.00984888
2,WW,0.05308287,0.05306276,-0.00002011,-0.00002011
3,2G,0.49556606,0.49059259,-0.00497347,-0.00497347
3,UBG,0.36939358,0.36235876,-0.00703482,-0.00703482
4,2UB,0.35814095,0.34776491,-0.01037604,-0.01037604
5,1UUGG,0.04311106,0.04109172,-0.00201934,-0.00201934


MV,Initial copies,Result copies,Copy delta,Weighted delta
1,0.68422336,0.67761241,-0.00661095,-0.00661095
2,1.07672311,1.05264569,-0.02407742,-0.02407742
3,0.86495964,0.85295136,-0.01200828,-0.01200828
4,0.35814095,0.34776491,-0.01037604,-0.01037604
5,0.12984787,0.12403429,-0.00581358,-0.00581358
6,0.10985099,0.10348371,-0.00636728,-0.00636728
9,0.00000000,0.00000000,+0.00000000,+0.00000000


Turn,Position,Initial copies,Result copies,Copy delta,Weighted delta
1,draw,0.03587743,0.03514963,-0.00072780,-0.00072780
1,play,0.03024361,0.02943870,-0.00080491,-0.00080491
2,draw,0.10882610,0.10578371,-0.00304239,-0.00304239
2,play,0.09137831,0.08815334,-0.00322497,-0.00322497
3,draw,0.19634771,0.19315177,-0.00319594,-0.00319594
3,play,0.16649048,0.16294646,-0.00354403,-0.00354403
4,draw,0.25862132,0.25204815,-0.00657317,-0.00657317
4,play,0.22115387,0.21392942,-0.00722445,-0.00722445
5,draw,0.31414736,0.30809875,-0.00604860,-0.00604860
5,play,0.27303788,0.26635527,-0.00668261,-0.00668261
